# lib

In [9]:
import os
import numpy as np
import pandas as pd
from nn_torch_ver.data_process import *
from nn_torch_ver.trainer import *
from nn_torch_ver.nn_model import *

from nn_torch_ver.fitness import *
from nn_torch_ver.dist import *
from nn_torch_ver.selection import *

# data

In [10]:
# -------- load wine dataset (local first) --------
RED_PATH = "winequality-red.csv"
WHITE_PATH = "winequality-white.csv"

if os.path.exists(RED_PATH) and os.path.exists(WHITE_PATH):
    print("✅ load from local files")
    red = pd.read_csv(RED_PATH, sep=";")
    white = pd.read_csv(WHITE_PATH, sep=";")
else:
    print("⬇️ download from UCI")
    red = pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        sep=";"
    )
    white = pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv",
        sep=";"
    )

red["type"] = "red"
white["type"] = "white"

df = pd.concat([red, white], ignore_index=True)

print(df.shape)
print(df.head())


⬇️ download from UCI
(6497, 13)
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality type  
0      9.4        5  red  


In [11]:
from torch.utils.data import Dataset, DataLoader

In [12]:
dm=WineNNDataManager(df)
wine_dataset = WineNNDataset(dm.get_numpy())
loader=DataLoader(wine_dataset,batch_size=32,shuffle=True,drop_last=True)

# train

In [13]:
encoder = navie_nn(
    dim=12,
    hidden_dims=[32, 32],
    out_dim=3,
    cdim=3,      
)
runner=exp_runner(encoder,loader,torch.optim.Adam(encoder.parameters()))

stat_norm tensor(28.7202)
k_norm tensor(0.8447)
mim_norm tensor(0.7276, grad_fn=<LinalgVectorNormBackward0>)
mim_norm tensor(0.7276, grad_fn=<LinalgVectorNormBackward0>)
loss_kernel 0.0005401660455390811
loss_stat 8.790902137756348
loss_per 4.395720958709717
total 4.395720958709717
stat_norm tensor(29.1270)
k_norm tensor(0.8203)
mim_norm tensor(0.2051, grad_fn=<LinalgVectorNormBackward0>)
mim_norm tensor(0.2051, grad_fn=<LinalgVectorNormBackward0>)
loss_kernel 0.00018268618441652507
loss_stat 8.969277381896973
loss_per 4.484730243682861
total 8.880451202392578
stat_norm tensor(32.8610)
k_norm tensor(0.7964)
mim_norm tensor(0.3215, grad_fn=<LinalgVectorNormBackward0>)
mim_norm tensor(0.3215, grad_fn=<LinalgVectorNormBackward0>)
loss_kernel 0.00018191809067502618
loss_stat 11.378994941711426
loss_per 5.68958854675293
total 14.570039749145508
stat_norm tensor(32.3405)
k_norm tensor(0.8053)
mim_norm tensor(0.5519, grad_fn=<LinalgVectorNormBackward0>)
mim_norm tensor(0.5519, grad_fn=<Linalg

In [14]:
decoder=decoding(32,3,16,14)
decoding_train(loader,encoder,decoder,torch.optim.Adam(decoder.parameters()),device,epochs=1)

[ep 0 | it 0] loss=5319.7925 score=5069.9199 kl=249.8727
[ep 0 | it 10] loss=1439.5287 score=1364.9259 kl=74.6028
[ep 0 | it 20] loss=439.2933 score=418.7084 kl=20.5849
[ep 0 | it 30] loss=167.4410 score=160.5979 kl=6.8431
[ep 0 | it 40] loss=91.5748 score=88.4270 kl=3.1477
[ep 0 | it 50] loss=45.1237 score=43.8751 kl=1.2486
[ep 0 | it 60] loss=15.5150 score=15.3147 kl=0.2003
[ep 0 | it 70] loss=9.2358 score=9.1601 kl=0.0757
[ep 0 | it 80] loss=5.0200 score=5.0502 kl=-0.0303
[ep 0 | it 90] loss=2.7310 score=2.7743 kl=-0.0433
[ep 0 | it 100] loss=1.7702 score=1.7882 kl=-0.0180
[ep 0 | it 110] loss=1.2395 score=1.2414 kl=-0.0020
[ep 0 | it 120] loss=0.9084 score=0.9580 kl=-0.0496
[ep 0 | it 130] loss=1.2307 score=1.2732 kl=-0.0425
[ep 0 | it 140] loss=0.6286 score=0.7219 kl=-0.0933
[ep 0 | it 150] loss=0.3178 score=0.4252 kl=-0.1074
[ep 0 | it 160] loss=0.2281 score=0.4569 kl=-0.2287
[ep 0 | it 170] loss=0.5345 score=0.6416 kl=-0.1071
[ep 0 | it 180] loss=0.5046 score=0.5185 kl=-0.0138
[

# inference

## verison encoding only

In [15]:
d1=next(iter(loader))

In [16]:
x,xo,q,t=d1['x'],d1['xo'],d1['quality'],d1['type']

In [17]:
g1=build_type_graph(t).to(device)
g2=build_type_graph(t).to(device)
g=torch.stack([g1,g2],dim=0)

In [18]:
m=encoder(x.float(),g.float())
m=m.requires_grad_(True)

In [19]:
reject=cov_mu_reject(m)

In [20]:
reject

tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 0, 1, 1], dtype=torch.int32)

## verison  decoding

In [21]:
score_hat=core_hat=torch.autograd.grad(
        decoder(m).sum(),m,retain_graph=True,create_graph=True)[0]

In [22]:
mu0=torch.ones(x.size(0))/32
cov0=torch.diag(mu0*(1-mu0))

In [23]:
mn,cn=update_prior_gaussian(mu0,cov0,m,score_hat)

In [24]:
llr = lrt_ni_vs_n(mn, cn)

chi_stat = 2 * llr

reject = (chi_stat > 3.84)

In [25]:
reject

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True])